# 00 - From noise to data: diffusion and flow-matching foundations

This notebook builds the visual vocabulary used by the rest of the course. The
audience is expected to know Python and vectors, but not stochastic calculus.

**Learning goals**

1. Recognize a forward noising path and a reverse generative path.
2. Read a time-dependent vector field as arrows that move points.
3. Explain the flow-matching training pair `(x_t, x_data - x_noise)`.
4. Distinguish a model, a denoiser, and a numerical sampler.

The toy dataset has three labeled clusters, but the generative model in later
notebooks will be trained **without labels**. Labels are retained only to test
post-hoc steering.


In [ ]:
import sys
from pathlib import Path

COURSE_DIR = Path.cwd()
if not (COURSE_DIR / "toy_course.py").exists():
    COURSE_DIR = Path("notebooks/toy_data")
sys.path.insert(0, str(COURSE_DIR.resolve()))

from IPython import get_ipython

ipython = get_ipython()
if ipython is not None:
    ipython.run_line_magic("matplotlib", "inline")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from toy_course import *

SEED = 2026
set_seed(SEED)
device = choose_device()
print(environment_summary(device))


## 1. A dataset we can see

Images live in thousands or millions of dimensions. Here every sample has only
two coordinates, so a scatter plot shows the complete data distribution. The
three colors play the role of semantic classes.


In [ ]:
clean, labels = sample_labeled_mixture(2400, device=device)
fig, ax = plt.subplots(figsize=(6, 5))
plot_labeled_points(clean, labels, ax=ax, title="Clean toy data")
ax.legend(frameon=False)
plt.show()


## 2. Forward noising

We use the linear path

$$x_t = t\,x_{data} + (1-t)\,\epsilon.$$

Here `epsilon` is Gaussian with standard deviation `s=1.8` in each coordinate.

- At `t=1`, the point is clean data.
- At `t=0`, the point is Gaussian noise.
- Lower `t` therefore means higher noise in this notebook series.

A diffusion model learns to undo a noising process. Flow matching learns the
velocity of a path between the same endpoints.


In [ ]:
set_seed(SEED)
clean_small, labels_small = sample_labeled_mixture(1500, device=device)
fixed_noise = sample_base(clean_small.shape[0], device=device)

times = [1.0, 0.70, 0.35, 0.08]
fig, axes = plt.subplots(1, len(times), figsize=(15, 3.8))
for ax, t_value in zip(axes, times):
    t = torch.full((clean_small.shape[0],), t_value, device=device)
    noisy = forward_noising(clean_small, t, fixed_noise)
    plot_labeled_points(noisy, labels_small, ax=ax, title=f"t={t_value:.2f}", alpha=0.42)
    ax.set_xlim(-5, 5)
    ax.set_ylim(-5, 5)
plt.tight_layout()
plt.show()


## 3. What flow matching predicts

Choose a noise point `x_noise`, a data point `x_data`, and a time `t`. Along the
straight conditional path, the target velocity is simply

$$u_t = x_{data} - x_{noise}.$$

Training repeatedly asks a neural network to predict this arrow from only
`(x_t, t)`. Because many endpoint pairs can pass near the same `x_t`, the
network learns an average velocity field, not a memorized arrow for each pair.


In [ ]:
set_seed(SEED + 1)
x_data, pair_labels = sample_labeled_mixture(24, device=device)
x_noise = sample_base(24, device=device)
t_value = 0.42
x_t = forward_noising(x_data, torch.full((24,), t_value, device=device), x_noise)
u_t = x_data - x_noise

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(x_noise[:, 0].cpu(), x_noise[:, 1].cpu(), s=18, color="#777777", label="noise endpoint")
ax.scatter(x_data[:, 0].cpu(), x_data[:, 1].cpu(), s=28, color="#0072B2", label="data endpoint")
ax.scatter(x_t[:, 0].cpu(), x_t[:, 1].cpu(), s=24, color="#E69F00", label="training state")
ax.quiver(
    x_t[:, 0].cpu(), x_t[:, 1].cpu(),
    u_t[:, 0].cpu(), u_t[:, 1].cpu(),
    angles="xy", scale_units="xy", scale=3.2, width=0.004, color="#222222", alpha=0.75,
)
ax.set_title("One flow-matching minibatch")
ax.set_aspect("equal")
ax.legend(frameon=False)
plt.show()


## 4. Four objects that must not be confused

| Object | Job | Example in this series |
|---|---|---|
| Noising path | Defines intermediate training states | `x_t = t*x_data + (1-t)*noise` |
| Neural model | Predicts a local quantity | velocity `v_theta(t, x_t)` |
| Denoiser | Estimates clean data from a noisy state | `D_theta(x_t,t)` |
| Sampler | Numerically follows model predictions | Euler, Heun, or RK4 |

Notebook `01` trains the velocity model. Notebook `02` converts its velocity
prediction into a denoised estimate and compares deterministic samplers.

**Check your understanding**

1. What is visible at high noise: exact class details or only coarse structure?
2. Why is `x_data - x_noise` available during training but unavailable during generation?
3. Which component changes when Euler is replaced by RK4: the trained model or the sampler?
